# Prediction Model - LOB

Goal: predict `Lob Associata` (LOB) and evaluate whether the task can be automated.

Workflow:
1. Data checks (3 key questions)
2. Fast manual prefix analysis
3. Rule baseline (no ML)
4. ML baselines (Decision Tree, Naive Bayes, KNN)
5. Interpretable rules and automation/manual-review policy

## 1) Setup

In [12]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)

## 2) Load Data

In [13]:
NOTEBOOK_DIR = Path.cwd()
CANDIDATES = [
    NOTEBOOK_DIR / 'DB for DTM Project.xlsx',
    NOTEBOOK_DIR.parent / 'DB for DTM Project.xlsx',
]
FILE_PATH = next((p for p in CANDIDATES if p.exists()), None)
if FILE_PATH is None:
    raise FileNotFoundError('DB for DTM Project.xlsx not found in current or parent folder')

SHEET = 'Associazioni Cod. Art. - LOB'
df = pd.read_excel(FILE_PATH, sheet_name=SHEET)
df.columns = [str(c).strip() for c in df.columns]

print('File:', FILE_PATH)
print('Shape:', df.shape)
print('Columns:', list(df.columns))

df.head(3)

File: c:\Users\sveta\Documents\vem-product\DB for DTM Project.xlsx
Shape: (22655, 5)
Columns: ['Codice Articolo', 'Lob Associata', 'Nome LOB', 'Inventario', 'Descrizione Articolo']


,Codice Articolo,Lob Associata,Nome LOB,Inventario,Descrizione Articolo
0,000050,1002,CABLAGGIO ALTERNATIVO,Inventario,PATCH CORD UTP 2/RJ45 CAT5E MT2
1,000081,1002,CABLAGGIO ALTERNATIVO,Inventario,cab - cf54/150ez passerella a filo 3m
2,‎000411AA230621QMANE,3004,CLOUD VARIE HW,Inventario,QUARKZMAN 30pz Ultra Sottile Telefono Leva Ape...


## 3) Define Core Columns

In [14]:
CODE_COL = 'Codice Articolo'
LOB_COL = 'Lob Associata'
LOB_NAME_COL = 'Nome LOB'
INV_COL = 'Inventario'
DESC_COL = 'Descrizione Articolo'

required = [CODE_COL, LOB_COL, INV_COL, DESC_COL]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

df_work = df[[c for c in [CODE_COL, LOB_COL, LOB_NAME_COL, INV_COL, DESC_COL] if c in df.columns]].copy()
for c in [CODE_COL, LOB_COL, INV_COL, DESC_COL]:
    df_work[c] = df_work[c].fillna('')

print('Working shape:', df_work.shape)
df_work.head(3)

Working shape: (22655, 5)


,Codice Articolo,Lob Associata,Nome LOB,Inventario,Descrizione Articolo
0,000050,1002,CABLAGGIO ALTERNATIVO,Inventario,PATCH CORD UTP 2/RJ45 CAT5E MT2
1,000081,1002,CABLAGGIO ALTERNATIVO,Inventario,cab - cf54/150ez passerella a filo 3m
2,‎000411AA230621QMANE,3004,CLOUD VARIE HW,Inventario,QUARKZMAN 30pz Ultra Sottile Telefono Leva Ape...


## 4) Step 1 - Data Checks (3 Questions)

Q1. One codice -> always one LOB?

Q2. Are there clear code patterns?

Q3. Is there a relationship between LOB and Inventario?

In [15]:
# Q1: One codice -> one LOB?
code_lob_uniques = (
    df_work.groupby(CODE_COL)[LOB_COL]
    .nunique(dropna=True)
    .reset_index(name='lob_nunique')
)

ambiguous_codes = code_lob_uniques[code_lob_uniques['lob_nunique'] > 1].copy()

print('Unique codici:', len(code_lob_uniques))
print('Codici with >1 LOB:', len(ambiguous_codes))
print('Share ambiguous:', f"{len(ambiguous_codes)/len(code_lob_uniques):.2%}")

ambiguous_codes.head(20)

Unique codici: 22655
Codici with >1 LOB: 0
Share ambiguous: 0.00%


,Codice Articolo,lob_nunique


In [16]:
# Helper prefix extractor for manual analysis
def extract_prefix(code: str, n: int = 5) -> str:
    text = str(code).upper().strip()
    text = re.sub(r'[^A-Z0-9]', '', text)
    if not text:
        return 'EMPTY'
    return text[:n]

df_work['prefix'] = df_work[CODE_COL].apply(extract_prefix)

# Q2: clear patterns in code prefixes?
prefix_lob = (
    df_work.groupby(['prefix', LOB_COL])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)

prefix_stats = prefix_lob.copy()
prefix_stats['prefix_total'] = prefix_stats.groupby('prefix')['count'].transform('sum')
prefix_stats['confidence'] = prefix_stats['count'] / prefix_stats['prefix_total']

prefix_top = (
    prefix_stats.sort_values(['prefix', 'confidence', 'count'], ascending=[True, False, False])
    .drop_duplicates('prefix')
    .rename(columns={LOB_COL: 'dominant_lob', 'count': 'dominant_count'})
    .sort_values(['confidence', 'prefix_total'], ascending=False)
)

print('Prefixes:', prefix_top.shape[0])
print('Strong prefixes (support>=20 and confidence>=0.9):', int(((prefix_top['prefix_total'] >= 20) & (prefix_top['confidence'] >= 0.9)).sum()))

prefix_top.head(30)

Prefixes: 10125
Strong prefixes (support>=20 and confidence>=0.9): 43


,prefix,dominant_lob,dominant_count,prefix_total,confidence
4317,CPAPS,6002,102,102,1.0
5405,FAS27,3014,43,43,1.0
2557,A1200,1006,36,36,1.0
2704,ACDC2,17001,33,33,1.0
4609,CVN07,1003,28,28,1.0
3130,ASA55,6001,27,27,1.0
4731,DG7GM,3002,27,27,1.0
4972,E3NMX,2016,27,27,1.0
9998,SWONT,3014,27,27,1.0
11100,WSC35,2002,26,26,1.0


In [17]:
# Q3: relationship LOB <-> Inventario
ct_count = pd.crosstab(df_work[LOB_COL], df_work[INV_COL])
ct_row_pct = pd.crosstab(df_work[LOB_COL], df_work[INV_COL], normalize='index').round(3)

print('LOB x Inventario (counts):')
display(ct_count.head(20))

print('LOB x Inventario (row percentages):')
display(ct_row_pct.head(20))

LOB x Inventario (counts):


Inventario,Assistenza,Inventario,Non in inventario
Lob Associata,,,
1,0,1,0
2,0,10,0
3,0,5,1
4,0,10,0
12,0,1,0
16,0,0,1
17,1,7,0
24,0,1,0
99,0,8,8


LOB x Inventario (row percentages):


Inventario,Assistenza,Inventario,Non in inventario
Lob Associata,,,
1,0.000,1.000,0.000
2,0.000,1.000,0.000
3,0.000,0.833,0.167
4,0.000,1.000,0.000
12,0.000,1.000,0.000
16,0.000,0.000,1.000
17,0.125,0.875,0.000
24,0.000,1.000,0.000
99,0.000,0.500,0.500


## 5) Step 2 - Fast Manual Prefix Analysis

Simple table: `prefix -> dominant LOB` with support and confidence.

In [18]:
manual_prefix_table = prefix_top[['prefix', 'dominant_lob', 'prefix_total', 'dominant_count', 'confidence']].copy()
manual_prefix_table = manual_prefix_table.sort_values(['confidence', 'prefix_total'], ascending=False)

manual_prefix_table.head(50)

,prefix,dominant_lob,prefix_total,dominant_count,confidence
4317,CPAPS,6002,102,102,1.0
5405,FAS27,3014,43,43,1.0
2557,A1200,1006,36,36,1.0
2704,ACDC2,17001,33,33,1.0
4609,CVN07,1003,28,28,1.0
3130,ASA55,6001,27,27,1.0
4731,DG7GM,3002,27,27,1.0
4972,E3NMX,2016,27,27,1.0
9998,SWONT,3014,27,27,1.0
11100,WSC35,2002,26,26,1.0


## 6) Step 3 - Rule Baseline (No ML)

Rule: `if prefix = X -> LOB = Y` (learned on train only).

In [19]:
rule_df = df_work[[CODE_COL, LOB_COL, 'prefix']].copy()
rule_df = rule_df[rule_df[LOB_COL].astype(str).str.strip() != '']

# Stratified split requires at least 2 samples per class.
lob_counts = rule_df[LOB_COL].value_counts()
rare_lobs = set(lob_counts[lob_counts < 2].index)

rule_df_model = rule_df[~rule_df[LOB_COL].isin(rare_lobs)].copy()
rule_df_rare = rule_df[rule_df[LOB_COL].isin(rare_lobs)].copy()

print('Total rows for rule baseline:', len(rule_df))
print('Rows excluded as rare LOB (<2 samples):', len(rule_df_rare))
print('Rare LOB classes excluded:', len(rare_lobs))

if rule_df_model[LOB_COL].nunique() < 2:
    raise ValueError('Not enough non-rare classes for stratified split in rule baseline.')

train_df, test_df = train_test_split(
    rule_df_model,
    test_size=0.2,
    random_state=42,
    stratify=rule_df_model[LOB_COL]
)

prefix_to_lob = (
    train_df.groupby('prefix')[LOB_COL]
    .agg(lambda s: s.value_counts().idxmax())
    .to_dict()
)

default_lob = train_df[LOB_COL].value_counts().idxmax()

test_pred_rule = test_df['prefix'].map(prefix_to_lob).fillna(default_lob)

rule_acc = accuracy_score(test_df[LOB_COL], test_pred_rule)
rule_f1_macro = f1_score(test_df[LOB_COL], test_pred_rule, average='macro')
rule_f1_weighted = f1_score(test_df[LOB_COL], test_pred_rule, average='weighted')

rule_metrics = pd.DataFrame([
    {'model': 'Prefix Rule Baseline', 'accuracy': rule_acc, 'f1_macro': rule_f1_macro, 'f1_weighted': rule_f1_weighted}
])
rule_metrics



Total rows for rule baseline: 22655
Rows excluded as rare LOB (<2 samples): 13
Rare LOB classes excluded: 13


,model,accuracy,f1_macro,f1_weighted
0,Prefix Rule Baseline,0.550453,0.417576,0.572871


## 7) Step 4 - ML Baselines for LOB

Models:
- Decision Tree (mandatory)
- Naive Bayes (text-based)
- KNN (optional, included here)

In [20]:
ml_df = df_work[[CODE_COL, DESC_COL, INV_COL, LOB_COL, 'prefix']].copy()
ml_df = ml_df[ml_df[LOB_COL].astype(str).str.strip() != '']
ml_df[DESC_COL] = ml_df[DESC_COL].fillna('')

# Reuse rare classes from previous step if available; otherwise compute here.
if 'rare_lobs' not in globals():
    _counts = ml_df[LOB_COL].value_counts()
    rare_lobs = set(_counts[_counts < 2].index)

ml_df_model = ml_df[~ml_df[LOB_COL].isin(rare_lobs)].copy()

# Text features without target leakage
ml_df_model['input_text'] = (
    'CODE_' + ml_df_model[CODE_COL].astype(str) + ' ' +
    'PREFIX_' + ml_df_model['prefix'].astype(str) + ' ' +
    'INV_' + ml_df_model[INV_COL].astype(str) + ' ' +
    ml_df_model[DESC_COL].astype(str)
)

X = ml_df_model['input_text']
y = ml_df_model[LOB_COL].astype(str)

if y.nunique() < 2:
    raise ValueError('Not enough non-rare classes for ML modeling.')

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

vectorizer = CountVectorizer(lowercase=True, min_df=5, max_features=6000, ngram_range=(1,2), binary=True)
X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)
feature_names = vectorizer.get_feature_names_out()

print('Rows excluded as rare LOB (<2 samples):', len(ml_df) - len(ml_df_model))
print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)
print('Classes (after rare-class exclusion):', y.nunique())



Rows excluded as rare LOB (<2 samples): 13
Train shape: (18113, 6000)
Test shape: (4529, 6000)
Classes (after rare-class exclusion): 104


In [21]:
models = {
    'Decision Tree': DecisionTreeClassifier(
        random_state=42,
        max_depth=35,
        min_samples_leaf=5,
        class_weight='balanced'
    ),
    'Naive Bayes': MultinomialNB(alpha=0.5),
    'KNN': KNeighborsClassifier(
        n_neighbors=7,
        weights='distance',
        metric='cosine',
        algorithm='brute'
    )
}

results = []
trained_models = {}
predictions = {}
probabilities = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)

    results.append({
        'model': name,
        'accuracy': accuracy_score(y_test, pred),
        'f1_macro': f1_score(y_test, pred, average='macro'),
        'f1_weighted': f1_score(y_test, pred, average='weighted')
    })

    trained_models[name] = model
    predictions[name] = pred
    probabilities[name] = proba

ml_metrics = pd.DataFrame(results).sort_values('f1_macro', ascending=False).reset_index(drop=True)

compare_df = pd.concat([rule_metrics, ml_metrics], ignore_index=True).sort_values('f1_macro', ascending=False).reset_index(drop=True)
compare_df

,model,accuracy,f1_macro,f1_weighted
0,KNN,0.653124,0.457618,0.648278
1,Prefix Rule Baseline,0.550453,0.417576,0.572871
2,Naive Bayes,0.656436,0.359921,0.636183
3,Decision Tree,0.164054,0.246868,0.171378


In [22]:
for name in ml_metrics['model']:
    print(f'\n===== {name} =====')
    print(classification_report(y_test, predictions[name], zero_division=0))


===== KNN =====
              precision    recall  f1-score   support

        1001       0.61      0.62      0.62       100
        1002       0.55      0.66      0.60       403
        1003       0.49      0.55      0.52       216
        1006       0.71      0.66      0.69        89
        1008       0.78      0.74      0.76        84
        1011       0.00      0.00      0.00         7
        1015       0.00      0.00      0.00        10
        1016       0.00      0.00      0.00         4
       15007       0.77      0.71      0.74        14
       15008       0.59      0.77      0.67        13
          17       0.00      0.00      0.00         2
       17001       0.67      0.70      0.69       202
       17005       0.52      0.33      0.41        36
       17007       0.00      0.00      0.00         5
       17008       0.00      0.00      0.00         3
       17009       0.38      0.45      0.41        33
       19001       1.00      0.50      0.67         2
          

## 8) Interpretable Patterns / Rules

### 8.1 Prefix Rules (human-readable)

In [28]:
interpretable_prefix_rules = manual_prefix_table[
    (manual_prefix_table['prefix_total'] >= 20) &
    (manual_prefix_table['confidence'] >= 0.90)
].copy()

interpretable_prefix_rules.head(100)

,prefix,dominant_lob,prefix_total,dominant_count,confidence
4317,CPAPS,6002,102,102,1.000000
5405,FAS27,3014,43,43,1.000000
2557,A1200,1006,36,36,1.000000
2704,ACDC2,17001,33,33,1.000000
4609,CVN07,1003,28,28,1.000000
3130,ASA55,6001,27,27,1.000000
4731,DG7GM,3002,27,27,1.000000
4972,E3NMX,2016,27,27,1.000000
9998,SWONT,3014,27,27,1.000000
11100,WSC35,2002,26,26,1.000000


### 8.2 Decision Tree rules and top features

In [29]:
dt = trained_models['Decision Tree']

imp = pd.DataFrame({
    'feature': feature_names,
    'importance': dt.feature_importances_
}).sort_values('importance', ascending=False)

print('Top feature importances:')
display(imp.head(30))

print('\nDecision Tree rules (truncated):')
print(export_text(dt, feature_names=list(feature_names), max_depth=4))

Top feature importances:


,feature,importance
4953,red hat,0.026142
1545,code_a,0.023915
3215,inventario cyber,0.023428
3571,meraki,0.023275
1760,code_rs,0.022367
1482,citrix,0.021766
1730,code_pan,0.021716
1915,copertura,0.020207
2498,forcepoint,0.018346
2714,hitachi,0.018237



Decision Tree rules (truncated):
|--- red hat <= 0.50
|   |--- code_a <= 0.50
|   |   |--- inventario cyber <= 0.50
|   |   |   |--- meraki <= 0.50
|   |   |   |   |--- code_rs <= 0.50
|   |   |   |   |   |--- truncated branch of depth 31
|   |   |   |   |--- code_rs >  0.50
|   |   |   |   |   |--- truncated branch of depth 4
|   |   |   |--- meraki >  0.50
|   |   |   |   |--- 1yr <= 0.50
|   |   |   |   |   |--- truncated branch of depth 19
|   |   |   |   |--- 1yr >  0.50
|   |   |   |   |   |--- truncated branch of depth 7
|   |   |--- inventario cyber >  0.50
|   |   |   |--- rnw <= 0.50
|   |   |   |   |--- act <= 0.50
|   |   |   |   |   |--- truncated branch of depth 7
|   |   |   |   |--- act >  0.50
|   |   |   |   |   |--- truncated branch of depth 2
|   |   |   |--- rnw >  0.50
|   |   |   |   |--- cgp cgc <= 0.50
|   |   |   |   |   |--- truncated branch of depth 2
|   |   |   |   |--- cgp cgc >  0.50
|   |   |   |   |   |--- class: 15008
|   |--- code_a >  0.50
|   |   

### 8.3 Naive Bayes class tokens

In [25]:
nb = trained_models['Naive Bayes']
nb_rows = []
for class_idx, cls in enumerate(nb.classes_):
    top_idx = np.argsort(nb.feature_log_prob_[class_idx])[::-1][:20]
    for rank, idx in enumerate(top_idx, start=1):
        nb_rows.append({
            'lob': cls,
            'rank': rank,
            'token': feature_names[idx],
            'log_prob': nb.feature_log_prob_[class_idx, idx]
        })

nb_tokens_df = pd.DataFrame(nb_rows)
nb_tokens_df

,lob,rank,token,log_prob
0,1001,1,inv_inventario,-2.898148
1,1001,2,lszh,-4.437340
2,1001,3,cat,-4.522757
3,1001,4,rj45,-4.602269
4,1001,5,utp,-4.799880
...,...,...,...,...
2075,99012,16,custodia,-8.704502
2076,99012,17,curva,-8.704502
2077,99012,18,d1 a1,-8.704502
2078,99012,19,d1,-8.704502


## 9) Automation vs Manual Review

In [26]:
best_model_name = ml_metrics.iloc[0]['model']
best_model = trained_models[best_model_name]
best_pred = predictions[best_model_name]
best_proba = probabilities[best_model_name]

max_proba = best_proba.max(axis=1)
conf_bucket = pd.cut(max_proba, bins=[-1, 0.65, 0.85, 1.0], labels=['low_manual_review', 'medium', 'high'])

policy_df = pd.DataFrame({
    'y_true': y_test.reset_index(drop=True),
    'y_pred': pd.Series(best_pred),
    'max_proba': max_proba,
    'bucket': conf_bucket
})
policy_df['correct'] = (policy_df['y_true'] == policy_df['y_pred']).astype(int)

bucket_summary = policy_df.groupby('bucket', observed=False).agg(
    n=('correct', 'size'),
    accuracy=('correct', 'mean')
).reset_index()
bucket_summary['share'] = bucket_summary['n'] / len(policy_df)

# Rule-based auto-cases
strong_prefixes = set(interpretable_prefix_rules['prefix']) if 'interpretable_prefix_rules' in globals() else set()
rule_auto_share = test_df['prefix'].isin(strong_prefixes).mean() if len(test_df) else 0.0

print('Best ML model:', best_model_name)
print('Share of strong prefix-rule cases in test:', f'{rule_auto_share:.2%}')

display(bucket_summary)

Best ML model: KNN
Share of strong prefix-rule cases in test: 6.40%


,bucket,n,accuracy,share
0,low_manual_review,2018,0.425173,0.445573
1,medium,749,0.736983,0.165379
2,high,1762,0.878547,0.389048


## 10) Final Conclusion Template

Fill from outputs above:
1. Can we automate LOB prediction?
2. Where is model/rule confidence high?
3. Where is manual review needed?

Suggested policy:
- Auto-assign when strong prefix rule exists OR ML confidence is high.
- Manual review for low-confidence and rare/ambiguous cases.

In [27]:
best = compare_df.iloc[0]
print('FINAL SUMMARY - LOB')
print(f"Best approach: {best['model']}")
print(f"Accuracy: {best['accuracy']:.4f}")
print(f"F1 macro: {best['f1_macro']:.4f}")
print(f"F1 weighted: {best['f1_weighted']:.4f}")
print(f"Rare LOB classes excluded from train/test: {len(rare_lobs)}")

if 'bucket_summary' in globals() and not bucket_summary.empty:
    high = bucket_summary[bucket_summary['bucket'] == 'high']
    low = bucket_summary[bucket_summary['bucket'] == 'low_manual_review']
    if len(high):
        print(f"High-confidence share: {high['share'].iloc[0]:.2%}, accuracy: {high['accuracy'].iloc[0]:.2%}")
    if len(low):
        print(f"Manual-review share (low confidence): {low['share'].iloc[0]:.2%}")

print('Use interpretable artifacts:')
print('- Prefix rules: interpretable_prefix_rules')
print('- Decision Tree rules: export_text output')
print('- Naive Bayes tokens: nb_tokens_df')


FINAL SUMMARY - LOB
Best approach: KNN
Accuracy: 0.6531
F1 macro: 0.4576
F1 weighted: 0.6483
Rare LOB classes excluded from train/test: 13
High-confidence share: 38.90%, accuracy: 87.85%
Manual-review share (low confidence): 44.56%
Use interpretable artifacts:
- Prefix rules: interpretable_prefix_rules
- Decision Tree rules: export_text output
- Naive Bayes tokens: nb_tokens_df
